In [28]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import numpy as np
import os
import time

In [29]:
# Constants
RANDOMSTATE = 1082
TRAIN_SIZE = 0.2
DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "XGBoost (SMOTE)"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../UAVATTACK_Results.csv"

In [30]:
# Load real data
df = pd.read_csv("../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols).values
labels = df['label']

In [31]:
# Data Preprocessing
if labels.dtype == object or labels.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(labels)
else:
    y = labels.values

# No scaling, just use raw X
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOMSTATE, stratify=y
)

print("\nApplying SMOTE to the training data...")
# Get class distribution
unique, counts = np.unique(y_train, return_counts=True)
class_counts = dict(zip(unique, counts))
majority_class = max(class_counts, key=class_counts.get)
count_of_majority_class = class_counts[majority_class]
print(f"Number of samples in the majority class ({majority_class}) in y_train: {count_of_majority_class}")

# Set up SMOTE sampling strategy for all minority classes
desired_strategy = {cls: count_of_majority_class for cls in unique if cls != majority_class}
smote = SMOTE(random_state=RANDOMSTATE, sampling_strategy=desired_strategy)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled training features shape: {X_train_resampled.shape}")
print(f"Resampled training target shape: {y_train_resampled.shape}")
print(f"Class distribution in resampled y_train:\n{pd.Series(y_train_resampled).value_counts(normalize=True)}")


Applying SMOTE to the training data...
Number of samples in the majority class (0) in y_train: 624
Resampled training features shape: (1248, 82)
Resampled training target shape: (1248,)
Class distribution in resampled y_train:
0    0.5
1    0.5
Name: proportion, dtype: float64


In [32]:
# Train the XGBoost model
print("\nTraining the XGBoost model...")

model = XGBClassifier(
    objective='binary:logistic',
    random_state=RANDOMSTATE,
    eval_metric='mae',
)
train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time


Training the XGBoost model...


In [35]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 99.90%
F1 Score: 0.9990

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.99      1.00       398

    accuracy                           1.00      2898
   macro avg       1.00      1.00      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [34]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(model.get_params()),
    "num_epochs": "N/A"
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../UAVATTACK_Results.csv
